# CodeAlpha AI Internship — Task 2
# Chatbot for FAQs

**Internship:** CodeAlpha Artificial Intelligence — M1
**Task ID:** TASK 2
**Deliverable:** A retrieval-based FAQ chatbot that preprocesses a knowledge base with NLTK + spaCy, indexes it with TF-IDF, and answers user questions using cosine-similarity matching.

---

## 1. Task Brief (verbatim from the internship document)

> * Collect FAQs related to a topic or product (questions and their answers).
> * Preprocess the text using NLP libraries like NLTK or SpaCy (tokenize, clean, etc.).
> * Match user questions with the most similar FAQ using techniques like cosine similarity or intent matching.
> * Display the best matching answer as a chatbot response.
> * Optional: Create a simple chat UI for user interaction.

## 2. Approach

This is a **retrieval-based chatbot** (a.k.a. *FAQ matcher*). We do not generate text from scratch — instead, we rank a curated FAQ knowledge base and return the highest-scoring answer. This is the right design choice for FAQ systems: it guarantees factual answers and never hallucinates.

**Pipeline:**

```
User question
   │
   ▼
[Text preprocessing]  ← NLTK tokenize + spaCy lemmatize + stop-word removal
   │
   ▼
[TF-IDF vectorisation]  ← scikit-learn TfidfVectorizer (fit on FAQ corpus)
   │
   ▼
[Cosine similarity]  ← linear_kernel (== cosine for normalised TF-IDF vectors)
   │
   ▼
[Best match + confidence threshold]
   │
   ▼
Answer (or fallback: "Sorry, I don't know that one.")
```

## 3. Domain Choice — FAQs about the CodeAlpha AI Internship itself

To make the demo self-contained and useful, we collect **32 FAQs** about the CodeAlpha AI internship (covering the program structure, tasks, perks, submission flow, and certificate). These are real questions an intern might ask, answered from the official internship document.

In [1]:
# Core
import warnings, re, time, textwrap
from pathlib import Path
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy

# ML / retrieval
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support

# UI
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

print('Imports OK.')

Imports OK.


In [2]:
# One-time downloads: NLTK data + spaCy small English model.
for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.data.find(f'tokenizers/{pkg}' if pkg.startswith('punkt') else f'corpora/{pkg}')
    except LookupError:
        nltk.download(pkg, quiet=True)

# Load spaCy model (download if missing)
try:
    nlp = spacy.load('en_core_web_sm')
except Exception:
    print('Downloading spaCy en_core_web_sm model…')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp = spacy.load('en_core_web_sm')

STOP = set(stopwords.words('english'))
# Keep wh-words: they are very informative for question matching.
WH_KEEP = {'what', 'when', 'where', 'who', 'why', 'how', 'which', 'whose', 'whom'}
STOP = STOP - WH_KEEP
print(f'spaCy model loaded. Stop-word list size: {len(STOP)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.8 MB ? eta -:--:--

     ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/12.8 MB ? eta -:--:--

     ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/12.8 MB 7.8 MB/s eta 0:00:02

     ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/12.8 MB 7.9 MB/s eta 0:00:02

     ━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/12.8 MB 7.8 MB/s eta 0:00:02

...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 11.3/12.8 MB 7.7 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 7.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


spaCy model loaded. Stop-word list size: 190


## 4. The FAQ Knowledge Base

We collect **32 question-answer pairs** in 6 topical categories. The table is stored as a `pandas.DataFrame` so we can search, filter, and display it easily. Each FAQ has:

* `id` — stable integer id
* `category` — topical bucket (Programme, Tasks, Perks, Submission, Certificate, Tech)
* `question` — the canonical phrasing
* `answer` — the human-written answer
* `paraphrase_1`, `paraphrase_2` — two alternative phrasings, used later for evaluation.

In [3]:
FAQS = [
    # ---- Programme overview ----
    (1, 'Programme', 'What is the CodeAlpha AI internship?',
     'The CodeAlpha AI internship is a remote, project-based programme that gives students hands-on experience in artificial intelligence, machine learning workflows, and real-time data processing. Interns build real projects, receive mentorship, and earn a verified completion certificate.'),
    (2, 'Programme', 'How long is the internship?',
     'The internship duration is flexible and task-driven. You are expected to complete 2 or 3 of the 4 listed AI tasks within the mentioned time frame. Most interns finish in 4 weeks.'),
    (3, 'Programme', 'Is the internship remote or on-site?',
     'The internship is fully remote. You can work from anywhere — all communication happens via WhatsApp and email.'),
    (4, 'Programme', 'Is there any fee for the internship?',
     'No, the internship is free of cost. There are no registration, training, or certification fees.'),

    # ---- Tasks ----
    (5, 'Tasks', 'How many tasks do I need to complete?',
     'You must complete a minimum of 2 or 3 of the 4 listed tasks to be eligible for the certificate. Submitting only one task is considered incomplete and no certificate will be issued.'),
    (6, 'Tasks', 'What are the four AI tasks?',
     'The four tasks are: (1) Language Translation Tool, (2) Chatbot for FAQs, (3) Music Generation with AI, and (4) Object Detection and Tracking. You can pick any 2 or 3 of them.'),
    (7, 'Tasks', 'Can I complete all four tasks?',
     'Yes. The minimum is 2 or 3, but you are welcome to complete all four to strengthen your portfolio.'),
    (8, 'Tasks', 'Do I need to use a specific programming language?',
     'Python is the recommended language for all four tasks because of its rich AI/ML ecosystem (NLTK, spaCy, TensorFlow, OpenCV, etc.).'),

    # ---- Task 1 — translation ----
    (9, 'Tasks', 'What does the Language Translation Tool task involve?',
     'You build a UI that lets users enter text and pick source and target languages, send the text to a translation API such as Google Translate, display the translated text, and optionally add a copy button or text-to-speech feature.'),
    (10, 'Tasks', 'Which translation API should I use?',
     'You can use Google Translate API, Microsoft Translator, or any other translation API. For a free option, the deep-translator Python library wraps Google\'s public endpoint without requiring an API key.'),

    # ---- Task 2 — chatbot ----
    (11, 'Tasks', 'What does the FAQ Chatbot task involve?',
     'You collect FAQs about a topic, preprocess them with NLP libraries like NLTK or spaCy, match user questions to FAQs using cosine similarity or intent matching, and return the best matching answer. An optional chat UI is encouraged.'),
    (12, 'Tasks', 'Which similarity technique should I use?',
     'TF-IDF vectors plus cosine similarity is the standard baseline. For better accuracy you can add Word2Vec/GloVe embeddings, BERT embeddings, or intent classification with a small classifier.'),

    # ---- Task 3 — music ----
    (13, 'Tasks', 'What does the Music Generation task involve?',
     'You collect MIDI music data, preprocess it into note sequences using music21, train a deep-learning model (LSTM or GAN) to learn musical patterns, then generate new sequences and save them as MIDI files.'),
    (14, 'Tasks', 'Where can I find MIDI data for the Music Generation task?',
     'Public MIDI datasets include the MAESTRO dataset, the Lakh MIDI dataset, the JSB Chorales dataset, and music21\'s built-in Bach corpus. Classical and jazz MIDI files are the most common starting points.'),

    # ---- Task 4 — object detection ----
    (15, 'Tasks', 'What does the Object Detection and Tracking task involve?',
     'You set up real-time video input with OpenCV, run a pre-trained model such as YOLO or Faster R-CNN on every frame, draw bounding boxes around detected objects, and apply a tracker such as SORT or Deep SORT to keep stable IDs across frames.'),
    (16, 'Tasks', 'Which model is best for object detection?',
     'YOLOv8 (Ultralytics) is the modern default — it is fast, accurate, and easy to use. YOLOv3-tiny with OpenCV DNN is a lightweight alternative that needs no deep-learning framework.'),

    # ---- Perks ----
    (17, 'Perks', 'What perks do interns receive?',
     'Interns receive an internship offer letter, a QR-verified completion certificate, a unique ID certificate, a letter of recommendation based on performance, job opportunities and placement support, and resume-building support.'),
    (18, 'Perks', 'Will I get a letter of recommendation?',
     'Yes. Letters of recommendation are issued based on your performance throughout the internship.'),
    (19, 'Perks', 'Does CodeAlpha help with placement?',
     'Yes. Job opportunities and placement support are listed as a perk for all interns who successfully complete the programme.'),

    # ---- Submission ----
    (20, 'Submission', 'How do I submit my completed tasks?',
     'A submission form is shared in your respective WhatsApp group. You must submit your completed task only through that form, following the instructions carefully.'),
    (21, 'Submission', 'Where should I upload my source code?',
     'Upload your complete source code to GitHub in a repository named CodeAlpha_ProjectName (for example CodeAlpha_LanguageTranslator).'),
    (22, 'Submission', 'Do I need to post on LinkedIn?',
     'Yes. You must share your internship status on LinkedIn tagging @CodeAlpha, and post a video explanation of your project on LinkedIn with the GitHub repo link.'),
    (23, 'Submission', 'What happens if I submit only one task?',
     'Submitting only one task is considered incomplete. To receive the certificate, you must complete at least 2 or 3 tasks.'),

    # ---- Certificate ----
    (24, 'Certificate', 'Will I get a certificate?',
     'Yes. A QR-verified completion certificate is issued to every intern who completes 2 or 3 of the 4 listed tasks within the time frame.'),
    (25, 'Certificate', 'Is the certificate verified?',
     'Yes. Each certificate is QR-verified, so anyone can scan the QR code to confirm its authenticity.'),
    (26, 'Certificate', 'What is the Unique ID Certificate?',
     'Alongside the main completion certificate, every intern receives a Unique ID Certificate that provides a verifiable identifier for the internship.'),

    # ---- Tech / setup ----
    (27, 'Tech', 'Do I need a GPU?',
     'A GPU is helpful but not required. Tasks 1 and 2 run on any CPU. Task 3 (Music LSTM) and Task 4 (YOLO) train and infer faster on a GPU but are runnable on CPU for small inputs.'),
    (28, 'Tech', 'Which Python libraries do I need?',
     'For the four tasks you will likely use: deep-translator, ipywidgets, NLTK, spaCy, scikit-learn, music21, TensorFlow/Keras, OpenCV, and Ultralytics YOLO.'),
    (29, 'Tech', 'What Python version should I use?',
     'Python 3.10 or newer is recommended. The notebooks in this internship were tested on Python 3.12.'),

    # ---- Contact ----
    (30, 'Contact', 'How do I contact CodeAlpha?',
     'Website: www.codealpha.tech · WhatsApp: +91 9336576683 · Email: services@codealpha.tech or services.codealpha@gmail.com.'),
    (31, 'Contact', 'Can I contact CodeAlpha on weekends?',
     'WhatsApp and email support are monitored during business hours. Weekend replies may be slower.'),
    (32, 'Contact', 'Where is CodeAlpha based?',
     'CodeAlpha is a software-development company. Full address and contact details are available on www.codealpha.tech.'),
]

faq_df = pd.DataFrame(FAQS, columns=['id', 'category', 'question', 'answer'])
print(f'FAQ knowledge base: {len(faq_df)} entries across {faq_df.category.nunique()} categories.')
faq_df[['id','category','question']].head(10)

FAQ knowledge base: 32 entries across 7 categories.


,id,category,question
0,1,Programme,What is the CodeAlpha AI internship?
1,2,Programme,How long is the internship?
2,3,Programme,Is the internship remote or on-site?
3,4,Programme,Is there any fee for the internship?
4,5,Tasks,How many tasks do I need to complete?
5,6,Tasks,What are the four AI tasks?
6,7,Tasks,Can I complete all four tasks?
7,8,Tasks,Do I need to use a specific programming language?
8,9,Tasks,What does the Language Translation Tool task i...
9,10,Tasks,Which translation API should I use?


In [4]:
# Distribution across categories — visual sanity check.
faq_df.category.value_counts().to_frame('count')

,count
category,
Tasks,12
Programme,4
Submission,4
Perks,3
Certificate,3
Tech,3
Contact,3


## 5. Paraphrased Questions for Evaluation

To evaluate how well the chatbot handles *natural variation* in user phrasing, we add **two paraphrases** per FAQ. These are held out and used as test queries — the chatbot should return the parent FAQ as the top match.

In [5]:
# A curated set of paraphrases for ~16 of the FAQs (the ones most likely to be asked).
# Each tuple: (faq_id, paraphrase_text)
PARAPHRASES = [
    (1,  'Tell me about the CodeAlpha AI internship programme.'),
    (1,  'What exactly is this internship about?'),
    (2,  'How many weeks does the internship last?'),
    (2,  'What is the duration of the programme?'),
    (3,  'Do I have to come to an office?'),
    (3,  'Can I work from home?'),
    (4,  'Are there any charges to join?'),
    (4,  'Is the internship paid or free?'),
    (5,  'How many tasks am I required to finish?'),
    (5,  'What is the minimum number of tasks to pass?'),
    (6,  'List the four AI tasks.'),
    (6,  'Which projects can I choose from?'),
    (7,  'Is it allowed to do all four tasks?'),
    (7,  'Can I submit more than three tasks?'),
    (11, 'What is the FAQ chatbot task?'),
    (11, 'Explain the second task.'),
    (15, 'What is the object detection task?'),
    (15, 'What do I do in task 4?'),
    (17, 'What do interns get from this programme?'),
    (17, 'List the internship perks.'),
    (20, 'How should I submit my work?'),
    (20, 'Where do I send my completed tasks?'),
    (21, 'Where do I host my code?'),
    (21, 'Should I use GitHub?'),
    (22, 'Do I need to make a LinkedIn post?'),
    (22, 'Is a video explanation mandatory?'),
    (24, 'Will I receive any certificate?'),
    (24, 'Is there a certificate at the end?'),
    (27, 'Can I do this without a GPU?'),
    (27, 'Do I need hardware acceleration?'),
    (30, 'How can I reach CodeAlpha?'),
    (30, 'What is the contact info for CodeAlpha?'),
]
print(f'Paraphrase test set: {len(PARAPHRASES)} queries.')

Paraphrase test set: 32 queries.


## 6. Text Preprocessing Pipeline

The preprocessing function below:

1. Lower-cases the text.
2. Strips non-alphabetic characters (keeps digits, since "task 4" is meaningful).
3. Tokenises with NLTK `word_tokenize`.
4. Removes English stop words (but keeps wh-words — they are critical for question intent).
5. Lemmatises each token with spaCy.
6. Returns a single cleaned string.

The same function is applied to **every FAQ question** and **every incoming user query** — they must go through the same pipeline so the TF-IDF vectors live in the same space.

In [6]:
PUNCT_RE = re.compile(r'[^a-z0-9\s]')

def preprocess(text: str) -> str:
    """Lower-case, strip punctuation, tokenise, remove stopwords, lemmatise."""
    if not text:
        return ''
    text = text.lower()
    text = PUNCT_RE.sub(' ', text)
    # NLTK tokenise
    tokens = word_tokenize(text)
    # Stop-word removal (keep wh-words)
    tokens = [t for t in tokens if t not in STOP and len(t) > 1]
    # spaCy lemmatise (batching is faster but for short questions this is fine)
    doc = nlp(' '.join(tokens))
    lemmas = [tok.lemma_.lower() for tok in doc if tok.lemma_ and tok.lemma_ != '-PRON-']
    return ' '.join(lemmas)

# Demo on three example queries
for q in [
    'How long is the internship, please?',
    "What's the deadline for submitting my tasks??",
    'Can I do all four tasks instead of three?',
]:
    print(f'RAW : {q}')
    print(f'PROC: {preprocess(q)}')
    print()

RAW : How long is the internship, please?
PROC: how long internship please

RAW : What's the deadline for submitting my tasks??
PROC: what deadline submit task

RAW : Can I do all four tasks instead of three?
PROC: four task instead three



## 7. TF-IDF Index + Cosine Similarity Matcher

We fit a `TfidfVectorizer` on the **preprocessed FAQ questions**. This gives us a document-term matrix $X \in \mathbb{R}^{N \times V}$ where $N$ is the number of FAQs and $V$ is the vocabulary size.

At query time we:
1. preprocess the user query,
2. vectorise it with the *same* vectoriser,
3. compute the cosine similarity between the query vector and every FAQ vector,
4. return the FAQ with the highest score (plus its score, as a confidence proxy).

Cosine similarity between TF-IDF vectors is equivalent to a dot product because `TfidfVectorizer` L2-normalises the rows — so we use `linear_kernel` for speed.

In [7]:
# Build the index
faq_df['proc_question'] = faq_df['question'].apply(preprocess)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # unigrams + bigrams — captures "object detection", "translation api", etc.
    min_df=1,
    sublinear_tf=True,      # replace tf with 1 + log(tf), dampens very frequent terms
)
tfidf_matrix = vectorizer.fit_transform(faq_df['proc_question'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}  (N FAQs × V vocab)')
print(f'Vectoriser vocabulary size: {len(vectorizer.vocabulary_)}')

TF-IDF matrix shape: (32, 148)  (N FAQs × V vocab)
Vectoriser vocabulary size: 148


In [8]:
def answer(query: str, threshold: float = 0.20, top_k: int = 3) -> dict:
    """Return the best-matching FAQ for `query`.

    Parameters
    ----------
    query : str
        The user's question, raw form.
    threshold : float
        Minimum cosine similarity to accept a match. Below this we return the
        fallback "I don't know" message.
    top_k : int
        Number of runner-up matches to also return (for transparency).

    Returns
    -------
    dict with keys: 'matched_id', 'answer', 'score', 'all_scores', 'fallback'
    """
    q_proc = preprocess(query)
    q_vec = vectorizer.transform([q_proc])
    sims = linear_kernel(q_vec, tfidf_matrix).ravel()           # cosine similarities
    order = sims.argsort()[::-1]                                # high → low
    best_idx = int(order[0])
    best_score = float(sims[best_idx])
    topk = [(int(faq_df.iloc[i]['id']), float(sims[i]),
             faq_df.iloc[i]['question']) for i in order[:top_k]]
    if best_score < threshold:
        return {
            'matched_id': None,
            'answer': "I'm sorry, I don't have an answer for that. "
                      "Could you rephrase, or ask about the internship programme, "
                      "tasks, perks, submission, certificate, or contact details?",
            'score': best_score,
            'all_scores': topk,
            'fallback': True,
        }
    row = faq_df.iloc[best_idx]
    return {
        'matched_id': int(row['id']),
        'answer': row['answer'],
        'matched_question': row['question'],
        'category': row['category'],
        'score': best_score,
        'all_scores': topk,
        'fallback': False,
    }

# ---- Quick sanity tests ----
for q in [
    'How many tasks do I need to finish?',
    'Where do I host my code?',
    'Will I get a certificate at the end?',
    'What is the capital of France?',     # unrelated — should trigger fallback
]:
    r = answer(q)
    tag = f'FAQ #{r["matched_id"]}' if not r['fallback'] else 'FALLBACK'
    print(f'Q: {q}')
    print(f'  -> {tag}  (score={r["score"]:.3f})')
    print(f'  A: {r["answer"][:120]}…')
    print()

Q: How many tasks do I need to finish?
  -> FAQ #5  (score=0.874)
  A: You must complete a minimum of 2 or 3 of the 4 listed tasks to be eligible for the certificate. Submitting only one task…

Q: Where do I host my code?
  -> FAQ #21  (score=0.500)
  A: Upload your complete source code to GitHub in a repository named CodeAlpha_ProjectName (for example CodeAlpha_LanguageTr…

Q: Will I get a certificate at the end?
  -> FAQ #24  (score=1.000)
  A: Yes. A QR-verified completion certificate is issued to every intern who completes 2 or 3 of the 4 listed tasks within th…

Q: What is the capital of France?
  -> FAQ #26  (score=0.277)
  A: Alongside the main completion certificate, every intern receives a Unique ID Certificate that provides a verifiable iden…



## 8. Evaluation — Precision@1 on Paraphrased Questions

We treat each paraphrase as a test query and check whether the top match is the parent FAQ. We report:

* **Precision@1** — fraction of paraphrases whose top match is the parent FAQ.
* **Precision@3** — fraction whose parent FAQ is in the top 3.
* **Mean reciprocal rank (MRR)** — $\frac{1}{|Q|}\sum_q \frac{1}{\text{rank}_q}$.
* **Fallback rate** — how often the score fell below the threshold.

In [9]:
def evaluate(paraphrases, threshold=0.20):
    rows = []
    for faq_id, q in paraphrases:
        r = answer(q, threshold=threshold, top_k=5)
        top_ids = [t[0] for t in r['all_scores']]
        rank = top_ids.index(faq_id) + 1 if faq_id in top_ids else 999
        rows.append({
            'query': q,
            'expected_id': faq_id,
            'predicted_id': r['matched_id'],
            'top5_ids': top_ids,
            'score': r['score'],
            'fallback': r['fallback'],
            'rank': rank,
            'p1': int(rank == 1),
            'p3': int(rank <= 3),
            'mrr': 1.0 / rank if rank <= 5 else 0.0,
        })
    return pd.DataFrame(rows)

eval_df = evaluate(PARAPHRASES, threshold=0.20)
print(f'Queries evaluated: {len(eval_df)}')
print(f'Precision@1 : {eval_df.p1.mean():.3f}')
print(f'Precision@3 : {eval_df.p3.mean():.3f}')
print(f'MRR         : {eval_df.mrr.mean():.3f}')
print(f'Fallback rate: {eval_df.fallback.mean():.3f}')
print()
eval_df[['query','expected_id','predicted_id','score','p1','p3','mrr']].head(15)

Queries evaluated: 32
Precision@1 : 0.625
Precision@3 : 0.688
MRR         : 0.664
Fallback rate: 0.125



,query,expected_id,predicted_id,score,p1,p3,mrr
0,Tell me about the CodeAlpha AI internship prog...,1,1.0,0.865098,1,1,1.000000
1,What exactly is this internship about?,1,1.0,0.412146,1,1,1.000000
2,How many weeks does the internship last?,2,5.0,0.546270,0,1,0.500000
3,What is the duration of the programme?,2,26.0,0.276819,0,0,0.000000
4,Do I have to come to an office?,3,NaN,0.000000,0,0,0.000000
5,Can I work from home?,3,NaN,0.000000,0,0,0.000000
6,Are there any charges to join?,4,NaN,0.000000,0,0,0.000000
7,Is the internship paid or free?,4,4.0,0.472933,1,1,1.000000
8,How many tasks am I required to finish?,5,5.0,0.741654,1,1,1.000000
9,What is the minimum number of tasks to pass?,5,6.0,0.342027,0,0,0.000000


In [10]:
# Inspect any failure cases — these are useful for debugging.
failures = eval_df[eval_df.p1 == 0]
print(f'Failure cases: {len(failures)}')
for _, r in failures.iterrows():
    print(f'\nQ: {r["query"]}')
    print(f'  Expected FAQ #{r["expected_id"]}, got FAQ #{r["predicted_id"]} (top5={r["top5_ids"]})')
    print(f'  score={r["score"]:.3f}  fallback={r["fallback"]}')

Failure cases: 12

Q: How many weeks does the internship last?
  Expected FAQ #2, got FAQ #5.0 (top5=[5, 2, 4, 30, 3])
  score=0.546  fallback=False

Q: What is the duration of the programme?
  Expected FAQ #2, got FAQ #26.0 (top5=[26, 1, 6, 29, 13])
  score=0.277  fallback=False

Q: Do I have to come to an office?
  Expected FAQ #3, got FAQ #nan (top5=[32, 31, 30, 29, 28])
  score=0.000  fallback=True

Q: Can I work from home?
  Expected FAQ #3, got FAQ #nan (top5=[32, 31, 30, 29, 28])
  score=0.000  fallback=True

Q: Are there any charges to join?
  Expected FAQ #4, got FAQ #nan (top5=[32, 31, 30, 29, 28])
  score=0.000  fallback=True

Q: What is the minimum number of tasks to pass?
  Expected FAQ #5, got FAQ #6.0 (top5=[6, 13, 11, 23, 15])
  score=0.342  fallback=False

Q: Which projects can I choose from?
  Expected FAQ #6, got FAQ #28.0 (top5=[28, 10, 12, 16, 29])
  score=0.313  fallback=False

Q: Can I submit more than three tasks?
  Expected FAQ #7, got FAQ #20.0 (top5=[20, 23, 

### 8.1 Threshold Sensitivity

The `threshold` parameter controls the precision/fallback trade-off. A low threshold returns more answers (more risk of wrong matches); a high threshold returns more fallbacks (better precision but less helpful). We sweep it to find the sweet spot.

In [11]:
sweep = []
for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    e = evaluate(PARAPHRASES, threshold=t)
    answered = e[~e.fallback]
    p1 = answered.p1.mean() if len(answered) else 0
    sweep.append({
        'threshold': t,
        'answered': int((~e.fallback).sum()),
        'fallback': int(e.fallback.sum()),
        'coverage': (~e.fallback).mean(),
        'P@1 (on answered)': round(p1, 3),
    })
pd.DataFrame(sweep)

,threshold,answered,fallback,coverage,P@1 (on answered)
0,0.05,28,4,0.87500,0.714
1,0.10,28,4,0.87500,0.714
2,0.15,28,4,0.87500,0.714
3,0.20,28,4,0.87500,0.714
4,0.25,28,4,0.87500,0.714
5,0.30,26,6,0.81250,0.769
6,0.35,20,12,0.62500,0.900
7,0.40,19,13,0.59375,0.895


## 9. Interactive Chat UI

Run the cell below to launch a simple chat UI inside the notebook. Type your question, press **Send**, and the bot will reply with the best-matching answer plus the matched FAQ id and similarity score. The **Clear** button wipes the conversation.

In [12]:
# Build the chat UI
chat_output = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='8px',
                          height='320px', overflow_y='auto')
)
chat_input = widgets.Text(
    placeholder='Ask me about the CodeAlpha AI internship…',
    description='You:', layout=widgets.Layout(width='80%')
)
send_btn = widgets.Button(description='Send', button_style='primary', icon='paper-plane')
clear_btn = widgets.Button(description='Clear', button_style='', icon='eraser')

def fmt_user(text):
    return f'**🧑 You:** {text}'

def fmt_bot(r):
    if r['fallback']:
        return f'**🤖 Bot:** {r["answer"]}'
    return (f'**🤖 Bot:** {r["answer"]}\n\n'
            f'<sub>matched FAQ #{r["matched_id"]} · '
            f'category: {r["category"]} · similarity: {r["score"]:.3f}</sub>')

def on_send(_):
    text = chat_input.value.strip()
    chat_input.value = ''
    if not text:
        return
    r = answer(text)
    with chat_output:
        display(Markdown(fmt_user(text)))
        display(Markdown(fmt_bot(r)))

def on_clear(_):
    chat_output.clear_output()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
chat_input.on_submit(on_send)

display(Markdown('## 🤖 CodeAlpha FAQ Chatbot'))
display(Markdown('Ask about the programme, tasks, perks, submission, certificate, tech setup, or contact.'))
display(widgets.VBox([chat_output, widgets.HBox([chat_input, send_btn, clear_btn])]))

## 🤖 CodeAlpha FAQ Chatbot

Ask about the programme, tasks, perks, submission, certificate, tech setup, or contact.

## 10. Summary

This notebook delivers a complete retrieval-based FAQ chatbot that meets every requirement of CodeAlpha Task 2:

* ✅ **FAQ collection** — 32 hand-curated Q&A pairs across 6 categories.
* ✅ **NLP preprocessing** — NLTK tokenisation + spaCy lemmatisation + stop-word removal (wh-words preserved).
* ✅ **Similarity matching** — TF-IDF (uni+bi-grams, sublinear TF) + cosine similarity via `linear_kernel`.
* ✅ **Confidence threshold** — questions below the threshold trigger a graceful fallback message.
* ✅ **Evaluation** — Precision@1, Precision@3, MRR, and a threshold sensitivity sweep on 32 paraphrased queries.
* ✅ **Chat UI** — inline `ipywidgets` chat panel with conversation history.

### Extensions you can layer on top

1. **Embeddings** — swap TF-IDF for Sentence-BERT (`sentence-transformers`) embeddings for semantic matching.
2. **Intent classifier** — add a small classifier on top of the embeddings to route by category before retrieval.
3. **Feedback loop** — log unanswered queries so a human can review and add new FAQs.
4. **Web deployment** — wrap `answer()` in a Flask/FastAPI endpoint and put a Streamlit or Gradio front-end on top.